In [10]:
import os
import sys

from huggingface_hub import hf_hub_download

import torch
from torchvision.transforms import Compose
import open_clip

import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType
from onnxruntime.quantization.shape_inference import quant_pre_process

In [17]:
def hf_download(destination: str):

    repo_id = "laion/CLIP-ViT-B-32-256x256-DataComp-s34B-b86K"
    filenames = ["open_clip_model.safetensors", "open_clip_config.json"]

    os.makedirs(destination, exist_ok=True)

    for file in filenames:
        hf_hub_download(
            repo_id=repo_id,
            filename=file,
            local_dir=destination
        )

    print(f"Model {repo_id} downloaded to {destination}/")

model_filename = "open_clip_model.safetensors"
config_filename = "open_clip_config.josn"
destination = "clip_model"
clean_cache = True

hf_download(destination)

Model laion/CLIP-ViT-B-32-256x256-DataComp-s34B-b86K downloaded to clip_model/


In [18]:
from transformers import AutoImageProcessor

processor = AutoImageProcessor.from_pretrained("laion/CLIP-ViT-B-32-256x256-DataComp-s34B-b86K")
processor

OSError: Can't load image processor for 'laion/CLIP-ViT-B-32-256x256-DataComp-s34B-b86K'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'laion/CLIP-ViT-B-32-256x256-DataComp-s34B-b86K' is the correct path to a directory containing a preprocessor_config.json file

In [ ]:
script_state = {
    "model_name": "ViT-B-32",
    "pretrained_name": os.path.join(
        destination,
        model_filename
    ),
    "onnx_model_path": os.path.join(
        destination,
        "clip.onnx"
    ),
    "onnx_preprocess_path": os.path.join(
        destination,
        "clip_prep.onnx"
    ),
    "onnx_model_shapes_path": os.path.join(
        destination,
        "clip_shapes.onnx"
    ),
    "quant_pre_model_path": os.path.join(
        destination,
        "clip_pre_quantized.onnx"
    ),
    "quant_model_path": os.path.join(
        destination,
        "clip_quantized.onnx"
    ),
    "preprocess_path": os.path.join(
        destination,
        "preprocess.onnx"
    ),
    "device": "cpu"
}

os.makedirs(destination, exist_ok=True)

In [6]:
print("Loading Clip...")

clip_model, _, preprocess = open_clip.create_model_and_transforms(
    script_state["model_name"],
    pretrained=script_state["pretrained_name"],
    device=script_state["device"]
)
clip_model.visual.eval()

print(clip_model)

Loading Clip...
CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-11): 12 x ResidualAttentionBlock(
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=768, out_features=3072, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((768,), eps=1e-05, el

In [7]:
clip_model

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-11): 12 x ResidualAttentionBlock(
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=768, out_features=3072, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((768,), eps=1e-05, elementwise_affine

In [ ]:
from open_clip import get_tokenizer

tokenizer = get_tokenizer("hf-hub:laion/CLIP-ViT-B-32-256x256-DataComp-s34B-b86K")
tokenizer("howdy")

tensor([[49406, 42216, 49407,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0]])

In [33]:
from open_clip import get_tokenizer

tokenizer = get_tokenizer("hf-hub:laion/CLIP-ViT-g-14-laion2B-s12B-b42K")
tokenizer("howdy")

tensor([[49406, 42216, 49407,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0]])

In [34]:
tokenizer

In [36]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("laion/CLIP-ViT-g-14-laion2B-s12B-b42K")
tokenizer("howdy")

{'input_ids': [49406, 42216, 49407], 'attention_mask': [1, 1, 1]}

In [37]:
tokenizer

CLIPTokenizer(name_or_path='laion/CLIP-ViT-g-14-laion2B-s12B-b42K', vocab_size=49408, model_max_length=77, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|startoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	49406: AddedToken("<|startoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	49407: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [39]:
from transformers import AutoTokenizer
from transformers.onnx import export, FeaturesManager

# get an onnx model by converting HuggingFace pretrained model
model_name = "bert-base-cased"
tokenizer_path = "clip_model/tokenizer.onnx"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = FeaturesManager.get_model_from_feature("default", model_name)
model_kind, model_onnx_config = FeaturesManager.check_supported_model_or_raise(model, feature="default")

ModuleNotFoundError: No module named 'transformers.onnx'

In [42]:
from onnxruntime_extensions import gen_processing_models

tokenizer_path = "clip_model/tokenizer.onnx"

print(f"Generating ONNX tokenizer at {tokenizer_path}")
tokenizer_model = gen_processing_models(tokenizer, pre_kwargs={}, post_kwargs={})[0]
with open(tokenizer_path, "wb") as f:
    f.write(tokenizer_model.SerializeToString())

Generating ONNX tokenizer at clip_model/tokenizer.onnx


AttributeError: CLIPTokenizer has no attribute encoder

In [ ]:
onnx_config = model_onnx_config(model.config)
export(tokenizer,
        model=model,
        config=onnx_config,
        opset=12,
        output=model_path)

In [ ]:
tokenizer.save_pretrained()

AttributeError: 'SimpleTokenizer' object has no attribute 'save_pretrained'

In [8]:
preprocess

Compose(
    Resize(size=224, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    MaybeConvertMode()
    MaybeToTensor()
    Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711))
)

In [11]:
class PreProcesModule(torch.nn.Module):
    def __init__(self, preprocess: Compose):
        super().__init__()
        self.preprocess = preprocess
    
    def forward(self, x: torch.Tensor):
        return self.preprocess(x)

clip_prep = PreProcesModule(preprocess)
clip_prep.eval()

PreProcesModule()

In [13]:
print("Exporting model to onnx format...")

input_tensor = torch.ones((2, 3, 224, 224), dtype=torch.float32)

torch.onnx.export(clip_model.visual,
                (input_tensor),
                script_state["onnx_model_path"],
                input_names = ['images'],
                output_names = ['embeddings'],
                dynamic_shapes=({0: torch.export.Dim.DYNAMIC},),
                external_data=False
                )

Exporting model to onnx format...
[torch.onnx] Obtain model graph for `VisionTransformer([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `VisionTransformer([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
Applied 64 of general pattern rewrite rules.
[torch.onnx] Optimize the ONNX graph... ✅


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 20},
            producer_name='pytorch',
            producer_version='2.11.0+cpu',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"images"<FLOAT,[s77,3,224,224]>
            ),
            outputs=(
                %"embeddings"<FLOAT,[s77,512]>
            ),
            initializers=(
                %"positional_embedding"<FLOAT,[50,768]>{TorchTensor(...)},
                %"proj"<FLOAT,[768,512]>{TorchTensor(...)},
                %"conv1.weight"<FLOAT,[768,3,32,32]>{TorchTensor(...)},
                %"ln_pre.weight"<FLOAT,[768]>{TorchTensor(...)},
                %"ln_pre.bias"<FLOAT,[768]>{TorchTensor(...)},
                %"transformer.resblocks.0.ln_1.weight"<FLOAT,[768]>{TorchTensor(...)},
                %"transformer.resblocks.0.ln_1.bias"<FLOAT,[768]>{TorchTensor(...)},
         

In [16]:
print("Exporting preprocess to onnx format...")

input_tensor = torch.ones((2, 3, 224, 224), dtype=torch.float32)

torch.onnx.export(clip_prep,
                (input_tensor),
                script_state["onnx_preprocess_path"],
                input_names = ['raw_images'],
                output_names = ['images'],
                dynamic_shapes=({0: torch.export.Dim.DYNAMIC, 1: torch.export.Dim.DYNAMIC},),
                external_data=False
                )

Exporting preprocess to onnx format...


W0401 14:11:44.726000 309630 torch/fx/experimental/symbolic_shapes.py:8255] Unable to find user code corresponding to {u0}


[torch.onnx] Obtain model graph for `PreProcesModule()` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `PreProcesModule()` with `torch.export.export(..., strict=False)`... ❌
[torch.onnx] Obtain model graph for `PreProcesModule()` with `torch.export.export(..., strict=True)`...





def forward(self, arg0_1: "f32[s77, s27, 224, 224]"):
    # File: /home/kojo/Code/ByItsCover/bic-library-search/glenv/lib/python3.13/site-packages/torchvision/transforms/transforms.py:394 in forward, code: return F.center_crop(img, self.size)
    alias: "f32[s77, s27, 224, 224]" = torch.ops.aten.alias.default(arg0_1);  arg0_1 = None
    
    # File: /home/kojo/Code/ByItsCover/bic-library-search/glenv/lib/python3.13/site-packages/torchvision/transforms/transforms.py:285 in forward, code: return F.normalize(tensor, self.mean, self.std, self.inplace)
    clone: "f32[s77, s27, 224, 224]" = torch.ops.aten.clone.default(alias);  alias = clone = None
    _tensor_constant0: "f32[3]" = self._tensor_constant0
    lift_fresh_copy: "f32[3]" = torch.ops.aten.lift_fresh_copy.default(_tensor_constant0);  _tensor_constant0 = lift_fresh_copy = None
    _tensor_constant1: "f32[3]" = self._tensor_constant1
    lift_fresh_copy_1: "f32[3]" = torch.ops.aten.lift_fresh_copy.default(_tensor_constant1);  _t

[torch.onnx] Obtain model graph for `PreProcesModule()` with `torch.export.export(..., strict=True)`... ❌


TorchExportError: Failed to export the model with torch.export. [96mThis is step 1/3[0m of exporting the model to ONNX. Next steps:
- Modify the model code for `torch.export.export` to succeed. Refer to https://pytorch.org/docs/stable/generated/exportdb/index.html for more information.
- Debug `torch.export.export` and submit a PR to PyTorch.
- Create an issue in the PyTorch GitHub repository against the [96m*torch.export*[0m component and attach the full error stack as well as reproduction scripts.

## Exception summary

<class 'torch.fx.experimental.symbolic_shapes.GuardOnDataDependentSymNode'>: Could not guard on data-dependent expression Eq(u0, 1) (unhinted: Eq(u0, 1)).  (Size-like symbols: none)

consider using data-dependent friendly APIs such as guard_or_false, guard_or_true and statically_known_true.
Caused by: (_export/non_strict_utils.py:1159 in __torch_function__)
For more information, run with TORCH_LOGS="dynamic"
For extended logs when we create symbols, also add TORCH

In [ ]:

print("Quantizing model...")

model = onnx.load(script_state["onnx_model_path"])
model = onnx.shape_inference.infer_shapes(model)
onnx.save(model, script_state["onnx_model_shapes_path"])

quant_pre_process(script_state["onnx_model_shapes_path"],
    script_state["quant_pre_model_path"],
    skip_optimization=False,
    skip_symbolic_shape=True,
    verbose=3)

quantize_dynamic(script_state["quant_pre_model_path"],
                                script_state["quant_model_path"],
                                weight_type=QuantType.QUInt8)

In [ ]:
if clean_cache:
    print("Cleaning up...")
    os.remove(script_state["pretrained_name"])

os.remove(script_state["onnx_model_path"])
os.remove(script_state["onnx_model_shapes_path"])
os.remove(script_state["quant_pre_model_path"])

if os.path.isfile(script_state["onnx_model_path"] + '.data'):
    os.remove(script_state["onnx_model_path"] + '.data')


print(f"Model {script_state["pretrained_name"]} quantized to {destination}/")